In [ ]:
!cd amazom2023
!wget https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/All_Beauty.jsonl.gz https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_All_Beauty.jsonl.gz https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Books.jsonl.gz https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Books.jsonl.gz https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Electronics.jsonl.gz
!cd ~

In [1]:
import json
import gzip
import pandas as pd


In [2]:
def to_int(text):
    if text is None:
        return None
    num = ''
    started = False
    has_dot = False

    for ch in text:
        if ch.isdigit():
            num += ch
            started = True

        elif ch in '.,' and started and not has_dot:
            num += '.'
            has_dot = True

        elif started:
            break

    if not num:
        return None
    return float(num)

In [3]:
def value_to_text(value):
    
    if value is None:
        return None

    if isinstance(value, list):
        parts = []

        for item in value:
            if item is None:
                continue
            
            text = value_to_text(item)
            if text:
                parts.append(text)

        return "; ".join(parts)

    if isinstance(value, dict):
        parts = []
        for key, val in value.items():
            if not key:
                continue

            if val is None:
                continue

            text = value_to_text(val)
            if text:
                parts.append(f"{key}: {text}")

        return "; ".join(parts)
    return str(value)

In [4]:
def extract_fields(line, fields):

    line = json.loads(line)
    wline = {}

    for field in fields:
        value = line.get(field)
        if field == 'price':
            try: 
                float(value)
            except:
                value = to_int(value)

        elif field == 'details':
            value = value_to_text(value)
            
        wline[field] = value

    return wline

In [5]:
def load(path,l_path, fields,i):
    with gzip.open(path,'rt') as file:
        k=0
        buf = []
        n=1
        for line in file:
            wline = extract_fields(line, fields)
            buf.append(wline)
            k+=1
            if k >= 1_000_000:
                df = pd.DataFrame(buf)
                df.to_parquet(f'{l_path}/part_{n}.parquet', index=False)
                n+=1
                k=0
                buf = []
        if len(buf) != 0:        
            df = pd.DataFrame(buf)
            df.to_parquet(f'{l_path}/part_{n}.parquet',index=False)



In [6]:
path = ['amazon2023/meta_All_Beauty.jsonl.gz', 'amazon2023/All_Beauty.jsonl.gz',
        'amazon2023/meta_Books.jsonl.gz', 'amazon2023/Books.jsonl.gz', 
        'amazon2023/meta_Electronics.jsonl.gz', 'amazon2023/Electronics.jsonl.gz']


path_to_l = ['data/All_Beauty/meta', 'data/All_Beauty/otz', 
             'data/Books/meta','data/Books/otz',
             'data/Electronic/meta','data/Electronic/otz' ]

fields = [['main_category','average_rating','parent_asin', 'title', 'details', 'price', 'categories', 'rating_number'],
          ['rating', 'verified_purchase', 'user_id', 'parent_asin', 'timestamp']]

In [7]:
for i in range(len(path)):
    load(path[i], path_to_l[i], fields[i%2], i)

In [8]:
data = pd.read_parquet('data/Books/meta')

In [9]:
data

,main_category,average_rating,parent_asin,title,details,price,categories,rating_number
0,Books,4.5,0701169850,Chaucer,Publisher: Chatto & Windus; First Edition (Jan...,8.23,"[Books, Literature & Fiction, History & Critic...",29
1,Books,5.0,0435088688,Notes from a Kidwatcher,"Publisher: Heinemann; First Edition (May 20, 1...",3.52,"[Books, Reference, Words, Language & Grammar]",1
2,Books,4.7,0316185361,Service: A Navy SEAL at War,"Publisher: Little, Brown and Company; 1st edit...",17.17,"[Books, Biographies & Memoirs, Leaders & Notab...",3421
3,Books,4.4,0545425573,Monstrous Stories #4: The Day the Mice Stood S...,Publisher: Scholastic Paperbacks; Reprint edit...,7.43,"[Books, Children's Books, Science Fiction & Fa...",40
4,Buy a Kindle,4.5,B00KFOP3RG,Parker & Knight,"Publication date: May 18, 2014; Language: Engl...",0.00,"[Books, Mystery, Thriller & Suspense, Thriller...",381
...,...,...,...,...,...,...,...,...
4448176,Books,4.3,1594483574,Please Excuse My Daughter,Publisher: Riverhead Books; Reprint edition (A...,36.06,"[Books, Biographies & Memoirs, Community & Cul...",69
4448177,Books,5.0,9719317051,Inside the Southeast Asian Kitchen: Foodlore a...,"Publisher: Art Post Asia (January 1, 2007); La...",75.00,"[Books, Cookbooks, Food & Wine, Regional & Int...",1
4448178,Books,4.9,0029051509,Origin of Negative Dialectics,Publisher: Free Press; Trade edition (December...,18.39,"[Books, Politics & Social Sciences, Philosophy]",16
4448179,Books,4.8,0925873039,Trails Illustrated National Parks Guadalupe Mo...,Language: English; ISBN 10: 0925873039; ISBN 1...,4.99,"[Books, Reference, Atlases & Maps]",121
